# Scenario Playground Step 17: Artifact Export and Reload

This notebook creates the versioned all-Ridge research bundle and proves that a fresh runtime can load it without retraining or reading the model matrix. Success means 12 fitted pipelines, exact 58-feature ordering, finite all-target outputs, zero neutral increments, matching explanations/confidence/warnings, valid hashes, and zero locked-final-test access.

Scope is artifact export and offline reload only. It does not run XGBoost, backend/frontend code, APIs, Supabase, or servers.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "scenario_modeling").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the InsightPulse project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.artifact_export import (
    DEFAULT_BUNDLE_DIRECTORY,
    EXAMPLE_SCENARIO,
    export_scenario_model_bundle,
    persist_fresh_process_evidence,
    persist_step17_evidence,
    run_fresh_process_smoke,
)
from scenario_modeling.src.scenario_runtime import (
    load_scenario_model_bundle,
    validate_bundle_hashes,
)

DEFAULT_BUNDLE_DIRECTORY

## 1. Export the frozen bundle

Export-time code fits the already-approved target-specific Ridge configurations once on the 464 authorized pretest rows. The saved runtime payload excludes target labels and the training-oriented engine.

In [ ]:
artifacts = export_scenario_model_bundle(DEFAULT_BUNDLE_DIRECTORY)
evidence_paths = persist_step17_evidence(artifacts)
{
    "bundle": artifacts["output_directory"],
    "bundle_version": artifacts["bundle_metadata"]["bundle_version"],
    "pipeline_count": 12,
    "hashes_valid": artifacts["hash_validation"]["all_hashes_match"],
    "reference_equivalent": artifacts["equivalence"]["equivalent"],
    "maximum_numeric_difference": artifacts["equivalence"]["maximum_absolute_numeric_difference"],
    "locked_final_test_access_count": artifacts["locked_final_test_access_count"],
}

## 2. Load and call the inference-only runtime

This is the same interface the team can use offline. No matrix or labels are supplied to the loader.

In [ ]:
engine = load_scenario_model_bundle(DEFAULT_BUNDLE_DIRECTORY)
result = engine.predict(
    EXAMPLE_SCENARIO,
    diagnostic_as_of_date=artifacts["reference_date"],
)
prediction_table = pd.DataFrame(
    [
        {
            "target": target,
            "unit": result["targets"][target]["unit"],
            "alpha": result["targets"][target]["selected_alpha"],
            "baseline": result["targets"][target]["baseline_prediction"],
            "scenario": result["targets"][target]["scenario_prediction"],
            "incremental": result["targets"][target]["incremental_scenario_effect"],
            "confidence": result["confidence"]["targets"][target]["overall_research_confidence"],
        }
        for target in engine.target_names
    ]
)
prediction_table

## 3. Fresh-process guard

The smoke process blocks every scikit-learn `fit` call and rejects every attempted read below `data/scenario_playground/`. It loads twice to verify deterministic predictions, explanations, confidence fields, and warnings.

In [ ]:
fresh = run_fresh_process_smoke(DEFAULT_BUNDLE_DIRECTORY)
fresh_evidence_path = persist_fresh_process_evidence(fresh)
{
    key: fresh[key]
    for key in (
        "pipeline_count",
        "fit_attempts",
        "training_data_reads",
        "target_label_reads",
        "all_outputs_finite",
        "neutral_incremental_exactly_zero",
        "reference_explanation_match",
        "reference_confidence_match",
        "reference_warnings_match",
        "reference_maximum_numeric_difference",
        "repeated_load_deterministic",
        "locked_final_test_access_count",
    )
}

## 4. Acceptance checks

These assertions are intentionally small and explicit so a failed handoff is visible in the notebook.

In [ ]:
hash_check = validate_bundle_hashes(DEFAULT_BUNDLE_DIRECTORY)
assert len(engine.pipelines) == 12
assert len(engine.feature_names) == 58
assert len(engine.state_feature_names) == 45
assert all(
    result["targets"][target]["naive_benchmark_substituted"] is False
    for target in engine.target_names
)
assert result["explanations"]["reconciliation"]["all_targets_passed"]
assert fresh["fit_attempts"] == 0
assert fresh["training_data_reads"] == 0
assert fresh["target_label_reads"] == 0
assert fresh["reference_mismatch_count"] == 0
assert fresh["locked_final_test_access_count"] == 0
assert hash_check["all_hashes_match"]
"STEP_17_ARTIFACT_EXPORT_COMPLETE"

## Result

The bundle is the notebook-to-integration boundary: the team can distribute the ignored version directory, install the pinned requirements, load the trusted joblib bundle, and call all 12 Ridge models without retraining. The predictions remain research-only, noncausal, and constrained by the bundled state date and the low/very-low confidence evidence.